# 📐 AquaVolt-AI: Physics-Informed Machine Learning (PIML)
This notebook demonstrates the foundational PIML architecture. Instead of a pure black-box neural network, we bound our AI predictions using physical laws—specifically, the FAO-56 Sigmoid Prior for Crop Coefficient ($K_c$) based on satellite NDVI.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')


## 📡 Load Live Satellite NDVI Data

In [ ]:
SHEET_ID = '1c2a-3t8fF2g_PX_0ape4ASTsbr5uX0Zb6YPzT8jtuN8'
csv_url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet=Sheet1'

df = pd.read_csv(csv_url, low_memory=False)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df = df.dropna(subset=['ndvi', 'kc'])
print(f"Loaded {len(df)} NDVI records.")


## 🧮 Plot the Physics-Informed Sigmoid Boundary

In [ ]:
ndvi_range = np.linspace(0, 1, 500)
kc_prior = 0.15 + 0.95 / (1 + np.exp(-12 * (ndvi_range - 0.4)))
kc_upper = np.clip(kc_prior + 0.15, 0.15, 1.20)
kc_lower = np.clip(kc_prior - 0.15, 0.15, 1.20)

fig, ax = plt.subplots(figsize=(10, 6), facecolor='#0e1117')
ax.set_facecolor('#1a1a2e')

ax.fill_between(ndvi_range, kc_lower, kc_upper, alpha=0.15, color='#4fc3f7', label='Physics Envelope (±0.15)')
ax.plot(ndvi_range, kc_prior, color='#4fc3f7', linewidth=2.5, label='FAO-56 Prior (Sigmoid)')

# Sample data points
sample = df.sample(min(500, len(df)))
ax.scatter(sample['ndvi'], sample['kc'], color='#ef5350', alpha=0.6, s=15, label='Live AI Predictions')

ax.set_xlabel('Satellite NDVI (Sentinel-2 / MODIS)', color='white')
ax.set_ylabel('Crop Coefficient (Kc)', color='white')
ax.set_title('Physics-Informed Kc Estimation (Real-Time)', color='white', fontweight='bold', fontsize=14)
ax.legend()
plt.show()
